<a href="https://colab.research.google.com/github/unmtransinfo/drugcentral-tools/blob/master/python/colab/DrugCentral_Inxight_Drugs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DrugCentral - Inxight Integration - DRUGS

Merging DC and Inxight drugs via UNII IDs.

Note, from Inxight website: "The full database contains approximately 4,500 drugs, including FDA-approved, previously approved, over-the-counter, and investigational small-molecule and peptide drugs." Thus we expect many Inxight drugs will not be present in DrugCentral, which is scoped specifically with
*only* approved drugs.

*   Inxight data downloaded from: https://drugs.ncats.io/downloads-public
*   DrugCentral data available from: https://drugcentral.org/download


## Drugs

In [1]:
import sys,os
import numpy as np
!pip install --upgrade pandas>=3.0.0
import pandas as pd
from google.colab import drive as colab_drive, auth as colab_auth
import gspread
from google.auth import default as g_auth_default

### Mount Google Drive

In [2]:
colab_drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DATADIR='/content/drive/My Drive/UNM/DrugCentral/data/'
print(f'Files in {DATADIR}')
for dirname, _, filenames in os.walk(f'{DATADIR}'):
  for filename in filenames:
    print(os.path.join(dirname, filename))

Files in /content/drive/My Drive/UNM/DrugCentral/data/
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_structures.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_targets.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight Metadata.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight_Db.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/inxight_activity_targets_drugcentral_candidate.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_target_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/inx_act_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/inx_act_mapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_target_mapped.gsheet


### Authenticate with Google Colab for Google Sheets API access

In [4]:
colab_auth.authenticate_user()
creds, _ = g_auth_default()
gc = gspread.authorize(creds)

### Read data from Google Sheets

*   Inxight Drugs
*   DrugCentral Drugs
*   DrugCentral Drug Xrefs



In [5]:
inx_sheet_url = 'https://docs.google.com/spreadsheets/d/1hqsjzcWH4m3SbLWoWga8O9IBmRmsI1tl62maItSIbN4/edit'
inx_drug = None; inx_targets = None;
try:
    inx_ss = gc.open_by_url(inx_sheet_url) # Open spreadsheet by URL
    inx_ws_drug = inx_ss.worksheet("frdb-drugs") # Select worksheets (specify or use get_worksheet(index))
    inx_drug = inx_ws_drug.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {inx_sheet_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [6]:
dc_sheet_drug_url = 'https://docs.google.com/spreadsheets/d/1TTUr6L_SVP7w_JQV4HBycR_KAndukrd411zI-Qf-Agw/edit'
dc_drug = None; dc_xref = None;
try:
    dc_drug_ss = gc.open_by_url(dc_sheet_drug_url) # Open spreadsheet by URL
    dc_ws_drug = dc_drug_ss.worksheet("dc2023_structures") # Select worksheets (specify or use get_worksheet(index))
    dc_ws_xref = dc_drug_ss.worksheet("dc2023_xrefs") # Select worksheets (specify or use get_worksheet(index))
    dc_drug = dc_ws_drug.get_all_values() # Get all values (list of lists)
    dc_xref = dc_ws_xref.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {dc_sheet_drug_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### Removing drugs missing UNIIs

In [7]:
df_inx_drug = pd.DataFrame(inx_drug[1:], columns=inx_drug[0])
print(f"Inxight Drugs: {df_inx_drug.shape}")
print(f"Inxight IDs: {df_inx_drug['compound_id'].nunique()}; UNIIs: {df_inx_drug['compound_unii'].nunique()}")
print(f"Inxight Drugs without UNIIs: {df_inx_drug[df_inx_drug['compound_unii'].isna() | df_inx_drug['compound_unii'].str.match(r'^\s*$')].shape[0]}")
df_inx_drug = df_inx_drug[df_inx_drug['compound_unii'].notna() & ~df_inx_drug['compound_unii'].str.match(r'^\s*$')]
display(df_inx_drug.sample(10))

Inxight Drugs: (4051, 3)
Inxight IDs: 4051; UNIIs: 3985
Inxight Drugs without UNIIs: 66


,compound_id,compound_name,compound_unii
454,740,SYROSINGOPINE,PPG46JF0EG
3252,21592,AMYDRICAINE,1801659K6K
3005,9951,TELCAGEPANT,D42O649ALL
707,1183,HYDROXYPROGESTERONE,21807M87J2
3745,46841,LATRUNCULIN B,LW7U308U7U
2383,6809,ALDOSTERONE,4964P6T9RB
3125,10216,PRACINOSTAT,GPO2JN4UON
522,859,CANTHARIDIN,IGL471WQ8P
3473,34280,MATRINE,N390W430AC
712,1188,ACRIVASTINE,A20F9XAI7W


In [8]:
df_inx_drug.rename(columns={"compound_id": "inx_compound_id", "compound_name": "inx_compound_name", "compound_unii": "inx_compound_unii"}, inplace=True)

In [9]:
df_inx_drug['inxight_drug_url'] = df_inx_drug['inx_compound_unii'].str.replace('^', r'https://drugs.ncats.io/drug/', regex=True)
df_inx_drug['inx_compound_unii'] = df_inx_drug['inx_compound_unii'].replace(r'^\s*$', np.nan)
display(df_inx_drug.sample(10))

,inx_compound_id,inx_compound_name,inx_compound_unii,inxight_drug_url
1897,2677,DIHYDROERGOCRISTINE,05D48LUM4Z,https://drugs.ncats.io/drug/05D48LUM4Z
3392,30264,FLUOROCYCLOPENTENYLCYTOSINE,0Z4A82I0JO,https://drugs.ncats.io/drug/0Z4A82I0JO
2865,8996,TERAMEPROCOL,53YET703F2,https://drugs.ncats.io/drug/53YET703F2
1030,1524,CLONAZEPAM,5PE9FDE8GB,https://drugs.ncats.io/drug/5PE9FDE8GB
2037,2951,BIMOCLOMOL,9IYF14814M,https://drugs.ncats.io/drug/9IYF14814M
649,1121,PHENTERMINE,C045TQL4WP,https://drugs.ncats.io/drug/C045TQL4WP
3078,10142,ENTOSPLETINIB,6I3O3W6O3B,https://drugs.ncats.io/drug/6I3O3W6O3B
1308,1877,"DOTHIEPIN, Z-",CZ377VWX9P,https://drugs.ncats.io/drug/CZ377VWX9P
2832,8847,OMECAMTIV MECARBIL,2M19539ERK,https://drugs.ncats.io/drug/2M19539ERK
1250,1805,GLICLAZIDE,G4PX8C4HKV,https://drugs.ncats.io/drug/G4PX8C4HKV


In [11]:
df_dc_drug = pd.DataFrame(dc_drug[1:], columns=dc_drug[0])
print(f"DC Structures: {df_dc_drug.shape}")
print(f"DC Structures DC_IDs: {df_dc_drug['id'].nunique()}")
display(df_dc_drug.head())

DC Structures: (4995, 8)
DC Structures DC_IDs: 4995


,id,name,cas_reg_no,smiles,inchikey,inchi,formula,molweight
0,5392,capmatinib,1029712-80-8,CNC(=O)C1=C(C=C(C=C1)C2=NN3C(=CN=C3N=C2)CC4=CC...,LIOLIMKSCNQPLV-UHFFFAOYSA-N,InChI=1S/C23H17FN6O/c1-25-22(31)18-6-5-16(11-1...,C23H17FN6O,412.428
1,5393,selpercatinib,2152628-33-4,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,XIIOFHFUYBLOLW-UHFFFAOYSA-N,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",C29H31N7O3,525.613
2,5394,ripretinib,1442472-39-0,CCN1C2=CC(=NC=C2C=C(C1=O)C3=CC(=C(C=C3Br)F)NC(...,CEFJVGZHQAGLHS-UHFFFAOYSA-N,InChI=1S/C24H21BrFN5O2/c1-3-31-21-12-22(27-2)2...,C24H21BrFN5O2,510.367
3,5377,molnupiravir,,CC(C)C(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O)N1C=...,HTNPEHXGEKVIHG-QCNRFFRDSA-N,InChI=1S/C13H19N3O7/c1-6(2)12(19)22-5-7-9(17)1...,C13H19N3O7,329.309
4,5395,fluoroestradiol F 18,94153-53-4,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)[1...,KDLLNMRYZGUVMA-ZYMZXAKXSA-N,InChI=1S/C18H23FO2/c1-18-7-6-13-12-5-3-11(20)8...,C18H23FO2,289.381


In [13]:
df_dc_xref = pd.DataFrame(dc_xref[1:], columns=dc_xref[0])
print(f"DC Xrefs: {df_dc_xref.shape}")
print(f"DC Xrefs DC_IDs: {df_dc_xref['struct_id'].nunique()}")
display(df_dc_xref.head())

DC Xrefs: (82230, 4)
DC Xrefs DC_IDs: 4995


,struct_id,xref_type,xref,dc_struct_name
0,3649,CHEBI,CHEBI:10001,visnadine
1,5100,CHEBI,CHEBI:10014,voacamine
2,1875,CHEBI,CHEBI:100147,nalidixic acid
3,2846,CHEBI,CHEBI:10023,voriconazole
4,659,CHEBI,CHEBI:100241,ciprofloxacin


### Xref types from DrugCentral

In [14]:
display(df_dc_xref['xref_type'].value_counts(sort=True))

xref_type
MMSL                           8036
SNOMEDCT_US                    7470
ChEMBL_ID                      7129
NDDF                           5575
UMLSCUI                        5244
UNII                           5185
PUBCHEM_CID                    5013
DRUGBANK_ID                    4368
CHEBI                          4302
INN_ID                         4232
KEGG_DRUG                      4051
VANDF                          3729
RXNORM                         3525
MESH_SUPPLEMENTAL_RECORD_UI    2909
SECONDARY_CAS_RN               2248
IUPHAR_LIGAND_ID               2106
NUI                            2044
MESH_DESCRIPTOR_UI             2014
VUID                           1790
PDB_CHEM_ID                    1260
Name: count, dtype: int64

### DrugCentral struct_id to UNII mappings

In [15]:
df_dc_unii = df_dc_xref[df_dc_xref['xref_type'] == 'UNII']
df_dc_unii.rename(columns={'struct_id':'dc_struct_id', 'xref': 'dc_unii'}, inplace=True)
df_dc_unii.drop(columns=['xref_type'], inplace=True)
df_dc_unii.drop_duplicates(inplace=True, ignore_index=True)
print(f"DrugCentral DC_IDs: {df_dc_unii['dc_struct_id'].nunique()}; UNIIs: {df_dc_unii['dc_unii'].nunique()}")
display(df_dc_unii.head())

DrugCentral DC_IDs: 4931; UNIIs: 5169


,dc_struct_id,dc_unii,dc_struct_name
0,132,001O2254AC,alphaprodine
1,3521,003N66TS6T,rasagiline
2,4769,00435Z54H1,dimethylaminopropionylphenothiazine
3,358,004F72P8F4,bethanechol
4,4988,005990WHZZ,deoxycholic acid


# Merge DC and Inxight via UNII IDs.

In [16]:
dc_inx_drug = pd.merge(df_dc_unii, df_inx_drug, left_on='dc_unii', right_on='inx_compound_unii', how='right')
display(dc_inx_drug.sample(10))

,dc_struct_id,dc_unii,dc_struct_name,inx_compound_id,inx_compound_name,inx_compound_unii,inxight_drug_url
2826,NaN,NaN,NaN,8832,NOOPEPT,4QBJ98683M,https://drugs.ncats.io/drug/4QBJ98683M
854,1511,YO1UK1S598,isradipine,1337,ISRADIPINE,YO1UK1S598,https://drugs.ncats.io/drug/YO1UK1S598
2809,NaN,NaN,NaN,8779,MBX-2982,B5TRY67L51,https://drugs.ncats.io/drug/B5TRY67L51
3477,NaN,NaN,NaN,34661,METHAMPHETAMINE SACCHARATE,K2J544L5HP,https://drugs.ncats.io/drug/K2J544L5HP
2391,NaN,NaN,NaN,6842,GABOXADOL,K1M5RVL18S,https://drugs.ncats.io/drug/K1M5RVL18S
3167,NaN,NaN,NaN,10292,ELOCALCITOL,2WDS5F2V6Q,https://drugs.ncats.io/drug/2WDS5F2V6Q
2462,768,5UX2SD1KE2,mercaptamine,7426,CYSTEAMINE,5UX2SD1KE2,https://drugs.ncats.io/drug/5UX2SD1KE2
3683,NaN,NaN,NaN,46312,AMG-517,172V4FBZ75,https://drugs.ncats.io/drug/172V4FBZ75
3203,NaN,NaN,NaN,10352,TARENFLURBIL,501W00OOWA,https://drugs.ncats.io/drug/501W00OOWA
784,854,O5CB12L4FN,diazoxide,1265,DIAZOXIDE,O5CB12L4FN,https://drugs.ncats.io/drug/O5CB12L4FN


In [17]:
print(f"Inxight UNIIs: {dc_inx_drug['inx_compound_unii'].nunique(dropna=True)}")
print(f"DC-Inx mapped UNIIs: {dc_inx_drug[dc_inx_drug['inx_compound_unii'].notna() & dc_inx_drug['dc_unii'].notna()].shape[0]}")

Inxight UNIIs: 3984
DC-Inx mapped UNIIs: 2351


Export file of mapped drugs.

In [18]:
dc_inx_drug[dc_inx_drug['inx_compound_unii'].notna() & dc_inx_drug['dc_unii'].notna()].to_csv(f"{DATADIR}/dc_inx_drug_mapped.tsv", sep='\t', index=False)